# Lesson 28 — Asyncio: Practice (AWARENESS, collapsed)

Work in `lessons/lesson-28-asyncio/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used here is taught
there. This is an AWARENESS lesson: two exercises, no hunt, no Challenge.
Answer E1 in its markdown cell BEFORE running the driver (memory-first,
same law as Quick Recall). Write E2 in `fanout.py` (R16 — no
`solution.py`), then run the **Driver:** cell.

You create: `fanout.py` (E2). Installs: none — stdlib `asyncio` only.


## E1 — Read async code before running it

## Intent
**Predict the output — the forgotten `await` and the blocking call are
silent.**

**Type:** REAL-WORLD — Easy — writes 0 files (answers + run only)

**Task**
Two snippets below. For EACH: predict (a) what prints, (b) what the
program actually did or didn't do, (c) why. Write your predictions in
the answer cell, THEN execute both snippets (paste the driver code into
a `.py` file or `python -c` — async won't run in plain notebook cells
reliably) and compare.

**Snippet 1 — forgotten await**

```python
import asyncio

async def shout(text):
    await asyncio.sleep(0.1)
    return text.upper()

async def main():
    result = shout("hello")      # no await here
    print(type(result).__name__)
    print(await result)

asyncio.run(main())
```

**Snippet 2 — blocking inside async**

```python
import asyncio, time

async def slow(tag):
    time.sleep(0.5)              # no await — plain sync sleep
    return f"{tag} done"

async def main():
    started = time.perf_counter()
    await asyncio.gather(slow("a"), slow("b"), slow("c"))
    print(f"{time.perf_counter() - started:.2f}s")

asyncio.run(main())
```

**Coverage** → lesson.md Pitfalls 1–2 + Core Concepts 1/4.

**Expected** (after you run each snippet yourself):
```text
Snippet 1: coroutine (first print) — and the second line proves the fetch DID run on await-of-a-coroutine
Snippet 2: ~1.50s — three blocking sleeps, zero overlap
```


### E1 answers — write your predictions HERE before running anything

**Snippet 1:**
- (a) Prints: coroutine func object \n HELLO
- (b) What ran / what didn't: last line who called the coroutine result with await will run the shout func and return HELLO; the result = shout("hello") just initiate the coroutine func since there is no await keyword

**Snippet 2:**
- (a) Prints: it should be ~ '1.5s'
- (b) What happened to the three tasks: it will ran one by one the slow a b c, but since the slow is not provide any await then it blocking each other

*(Then run both, paste the observed output below, and note any surprise.)*

**Observed:**
S1: printed coroutine / HELLO — matches prediction.
S2: printed 1.50s — matches prediction (~1.5s). No surprises


In [ ]:
import subprocess, sys, tempfile, pathlib

S1 = '''import asyncio

async def shout(text):
    await asyncio.sleep(0.1)
    return text.upper()

async def main():
    result = shout("hello")      # no await here
    print(type(result).__name__)
    print(await result)

asyncio.run(main())'''

S2 = '''import asyncio, time

async def slow(tag):
    time.sleep(0.5)              # no await — plain sync sleep
    return f"{tag} done"

async def main():
    started = time.perf_counter()
    await asyncio.gather(slow("a"), slow("b"), slow("c"))
    print(f"{time.perf_counter() - started:.2f}s")

asyncio.run(main())'''

with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    p1, p2 = td / "s1.py", td / "s2.py"
    p1.write_text(S1, encoding="utf-8")
    p2.write_text(S2, encoding="utf-8")
    r1 = subprocess.run([sys.executable, str(p1)], capture_output=True, text=True)
    assert r1.returncode == 0, r1.stderr
    assert r1.stdout.splitlines() == ["coroutine", "HELLO"], r1.stdout
    r2 = subprocess.run([sys.executable, str(p2)], capture_output=True, text=True)
    assert r2.returncode == 0, r2.stderr
    secs = float(r2.stdout.strip().removesuffix("s"))
    assert secs >= 1.4, f"blocking expected ~1.5s, got {secs}"   # no overlap
    assert secs < 2.0, f"unexpectedly slow machine: {secs}"
print("E1 OK")

i think there is nothing wrong with my answer, but i have a question.. is it the right reason? explain to me thanks


S1: printed coroutine / HELLO — matches prediction.
S2: printed 1.50s — matches prediction (~1.5s). No surprises


## E2 — Fan out with gather

## Intent
**The whole lesson in one function: three waits, one total.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `fanout.py`, define `async def fetch(name: str, delay: float) -> str`
(awaits `asyncio.sleep(delay)`, returns `f"{name} done"`) and
`async def fetch_all(pairs) -> list[str]` that runs every
`(name, delay)` **concurrently** via `asyncio.gather`, returning
results in argument order.

**File tree**

- `fanout.py` (`fetch`, `fetch_all`)

**Interface contract**

- `await fetch_all([("a", 0.4), ("b", 0.4), ("c", 0.4)])` →
  `["a done", "b done", "c done"]` — order = arguments, not completion.
- Total wall time under a second for the three 0.4s fetches (sequential
  awaits would take 1.2s — the driver pins the shape).

**Constraints**

- **`fetch_all` must `gather` — sequential awaits fail the timing pin.**
- **Results in argument order.**

**Coverage** → lesson.md Core Concepts 1–3 + the worked example.

**Expected** (Driver cell):
```text
E2 OK
```


In [ ]:
import asyncio, time
import fanout as f

got = asyncio.run(f.fetch_all([("a", 0.4), ("b", 0.4), ("c", 0.4)]))
assert got == ["a done", "b done", "c done"], got

started = time.perf_counter()
asyncio.run(f.fetch_all([("x", 0.4), ("y", 0.4), ("z", 0.4)]))
elapsed = time.perf_counter() - started
assert elapsed < 0.9, f"sequential awaits suspected: {elapsed:.2f}s"

one = asyncio.run(f.fetch_all([("solo", 0.1)]))
assert one == ["solo done"], one
print("E2 OK")

btw i ran it in script.py and its work not like the jupyter one, so its done